# Session 7: Multi-Agent Systems
**Duration:** 1.5 Hours | **Day 2** | **Hands-On Workshop**

---

## Learning Objectives
By the end of this session, participants will:
1. Understand multi-agent collaboration patterns
2. Design task decomposition strategies
3. Implement the Planner-Executor-Reviewer architecture
4. Build an AI Research Assistant using multiple agents
5. Understand orchestration patterns for autonomous workflows

---

## Prerequisites
- Completed Sessions 5 and 6 (Agents + MCP)
- Ollama running with `llama3.2`
- Packages: `pip install langchain langchain-ollama`

---
## Part 1: Multi-Agent Architecture (Theory - 20 min)

### 1.1 Why Multi-Agent?

Single agents have limitations:
- **Context window limits**: One agent cannot hold everything
- **Specialization**: Different tasks need different expertise
- **Quality**: One agent reviewing its own work rarely catches errors
- **Complexity**: Large tasks benefit from divide-and-conquer

### 1.2 Multi-Agent Patterns

```
Pattern 1: SEQUENTIAL PIPELINE
  Agent 1 (Research) --> Agent 2 (Analyze) --> Agent 3 (Write) --> Output
  Each agent passes its output to the next.

Pattern 2: HIERARCHICAL (Boss-Worker)
  Orchestrator Agent
       |--- Worker Agent 1 (Research)
       |--- Worker Agent 2 (Code)
       |--- Worker Agent 3 (Review)
  Boss delegates tasks, collects results, makes decisions.

Pattern 3: DEBATE/ADVERSARIAL
  Agent A (Proponent) <---> Agent B (Critic)
  Agents argue until consensus. Improves quality.

Pattern 4: COLLABORATIVE (Swarm)
  Agent 1 <---> Agent 2 <---> Agent 3
  All agents share a workspace and contribute freely.
```

### 1.3 The Planner-Executor-Reviewer Architecture

This is the most practical pattern for production systems:

```
User Request
    |
    v
[PLANNER Agent]
  - Breaks down the task into subtasks
  - Assigns subtasks to appropriate agents
  - Creates execution plan
    |
    v
[EXECUTOR Agents] (one or more)
  - Agent 1: Execute subtask 1
  - Agent 2: Execute subtask 2
  - Agent 3: Execute subtask 3
    |
    v
[REVIEWER Agent]
  - Reviews all outputs
  - Checks for quality and consistency
  - Requests revisions if needed
    |
    v
Final Output
```

---
## Part 2: Building Individual Agents (Hands-On - 20 min)

Let's create specialized agents that will work together.

In [ ]:
# 2.1 Setup
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
import json
from datetime import datetime

# Initialize LLM
llm = ChatOllama(model="llama3.2", temperature=0.3)

print("LLM initialized for multi-agent system!")

In [ ]:
# 2.2 Agent Base Class

class Agent:
    """Base class for specialized agents."""

    def __init__(self, name: str, role: str, system_prompt: str, llm):
        self.name = name
        self.role = role
        self.llm = llm
        self.system_prompt = system_prompt
        self.history = []

    def run(self, task: str, context: str = "") -> str:
        """Execute the agent on a task."""
        messages = [
            SystemMessage(content=self.system_prompt),
        ]

        if context:
            messages.append(HumanMessage(content=f"Context from previous agents:\n{context}\n\nYour task: {task}"))
        else:
            messages.append(HumanMessage(content=task))

        response = self.llm.invoke(messages)
        result = response.content

        # Log the interaction
        self.history.append({
            "task": task,
            "context_provided": bool(context),
            "result_length": len(result),
            "timestamp": datetime.now().isoformat()
        })

        return result

    def __repr__(self):
        return f"Agent(name={self.name}, role={self.role})"

print("Agent base class defined!")

In [ ]:
# 2.3 Create Specialized Agents

# Agent 1: Research Agent
researcher = Agent(
    name="Researcher",
    role="Research and gather information",
    system_prompt="""You are a thorough research agent. Your job is to:
1. Analyze the given topic deeply
2. Identify key aspects, features, and concepts
3. Find relevant examples and use cases
4. Present findings in a structured format

Output format:
## Research Findings
### Key Concepts
- [concept 1]: [explanation]
- [concept 2]: [explanation]

### Important Details
- [detail 1]
- [detail 2]

### Examples/Use Cases
- [example 1]
- [example 2]

Be thorough but concise. Focus on factual, useful information.""",
    llm=llm
)

# Agent 2: Summarizer Agent
summarizer = Agent(
    name="Summarizer",
    role="Create clear, concise summaries",
    system_prompt="""You are an expert summarizer. Your job is to:
1. Take research findings and create a well-organized summary
2. Highlight the most important points
3. Make information accessible and easy to understand
4. Organize content logically with clear sections

Output format:
## Summary
### Overview
[2-3 sentence overview]

### Key Points
1. [point with explanation]
2. [point with explanation]
3. [point with explanation]

### Practical Implications
[How this information can be applied]

Keep it under 300 words. Every sentence must add value.""",
    llm=llm
)

# Agent 3: Critic Agent
critic = Agent(
    name="Critic",
    role="Review and improve quality",
    system_prompt="""You are a critical reviewer. Your job is to:
1. Evaluate the quality and accuracy of provided content
2. Identify gaps, errors, or weak arguments
3. Suggest specific improvements
4. Rate the overall quality

Output format:
## Critical Review
### Quality Rating: [1-10]

### Strengths
- [strength 1]
- [strength 2]

### Weaknesses / Gaps
- [issue 1]: [specific suggestion to fix]
- [issue 2]: [specific suggestion to fix]

### Missing Information
- [what should be added]

### Revised Version
[If quality < 7, provide an improved version incorporating your suggestions]

Be constructive but honest. Provide specific, actionable feedback.""",
    llm=llm
)

# Agent 4: Final Output Generator
output_generator = Agent(
    name="OutputGenerator",
    role="Produce polished final output",
    system_prompt="""You are a professional content editor. Your job is to:
1. Take the research, summary, and review feedback
2. Produce a polished, final document
3. Incorporate all valid feedback from the critic
4. Ensure consistency, clarity, and completeness

Output format:
## [Topic Title]

[Professional, well-structured content that synthesizes all inputs]

### Key Takeaways
- [takeaway 1]
- [takeaway 2]
- [takeaway 3]

The final output should read as a polished, standalone document.
It should be informative, accurate, and well-organized.""",
    llm=llm
)

print("Created 4 specialized agents:")
for agent in [researcher, summarizer, critic, output_generator]:
    print(f"  - {agent.name} ({agent.role})")

---
## Part 3: Multi-Agent Orchestrator (Hands-On - 25 min)

Now let's build the orchestrator that coordinates all agents.

In [ ]:
# 3.1 Sequential Pipeline Orchestrator

class MultiAgentPipeline:
    """Orchestrates multiple agents in a sequential pipeline."""

    def __init__(self, agents: list, name: str = "Pipeline"):
        self.agents = agents
        self.name = name
        self.execution_log = []

    def run(self, initial_task: str, verbose: bool = True) -> dict:
        """Execute the pipeline on a task."""
        if verbose:
            print(f"\n{'='*60}")
            print(f"  MULTI-AGENT PIPELINE: {self.name}")
            print(f"  Task: {initial_task}")
            print(f"  Agents: {' --> '.join(a.name for a in self.agents)}")
            print(f"{'='*60}")

        context = ""  # Accumulated context from previous agents
        results = {}  # Store each agent's output

        for i, agent in enumerate(self.agents):
            step = i + 1

            if verbose:
                print(f"\n{'---'*15}")
                print(f"  Step {step}/{len(self.agents)}: {agent.name} ({agent.role})")
                print(f"{'---'*15}")

            # Run the agent
            if i == 0:
                # First agent gets the original task
                result = agent.run(initial_task)
            else:
                # Subsequent agents get accumulated context
                result = agent.run(initial_task, context=context)

            # Store result
            results[agent.name] = result
            context += f"\n\n=== Output from {agent.name} ===\n{result}"

            # Log
            self.execution_log.append({
                "step": step,
                "agent": agent.name,
                "output_length": len(result)
            })

            if verbose:
                # Show first 300 chars of output
                preview = result[:300] + "..." if len(result) > 300 else result
                print(f"\n{preview}")

        if verbose:
            print(f"\n{'='*60}")
            print(f"  PIPELINE COMPLETE")
            print(f"  Total agents: {len(self.agents)}")
            print(f"{'='*60}")

        return results

print("MultiAgentPipeline class defined!")

In [ ]:
# 3.2 Build the AI Research Assistant Pipeline

research_pipeline = MultiAgentPipeline(
    agents=[researcher, summarizer, critic, output_generator],
    name="AI Research Assistant"
)

# Run on a topic
results = research_pipeline.run(
    "Explain Retrieval Augmented Generation (RAG) and its importance in modern AI applications"
)

In [ ]:
# 3.3 View the final output
print("=" * 60)
print("  FINAL OUTPUT")
print("=" * 60)
print(results["OutputGenerator"])

In [ ]:
# 3.4 View execution log
print("=== Execution Log ===")
for log in research_pipeline.execution_log:
    print(f"  Step {log['step']}: {log['agent']} -- produced {log['output_length']} chars")

---
## Part 4: Advanced -- Iterative Refinement Loop (Hands-On - 15 min)

Add a feedback loop where the critic can send work back for revision.

In [ ]:
# 4.1 Iterative Multi-Agent System

class IterativeAgentSystem:
    """Multi-agent system with iterative refinement.
    The critic can request revisions until quality is satisfactory."""

    def __init__(self, researcher, writer, critic, llm, max_revisions: int = 2):
        self.researcher = researcher
        self.writer = writer
        self.critic = critic
        self.llm = llm
        self.max_revisions = max_revisions

    def _extract_quality_score(self, review: str) -> int:
        """Extract quality rating from critic's review."""
        # Try to find a number after 'Rating:' or similar
        import re
        match = re.search(r'Rating:\s*(\d+)', review)
        if match:
            return int(match.group(1))
        # Default to 5 if we cannot find a rating
        return 5

    def run(self, topic: str, quality_threshold: int = 7) -> dict:
        """Run the iterative multi-agent system."""
        print(f"\n{'='*60}")
        print(f"  ITERATIVE MULTI-AGENT SYSTEM")
        print(f"  Topic: {topic}")
        print(f"  Quality threshold: {quality_threshold}/10")
        print(f"  Max revisions: {self.max_revisions}")
        print(f"{'='*60}")

        # Step 1: Research
        print(f"\n[1] RESEARCHER is gathering information...")
        research = self.researcher.run(topic)
        print(f"    Done. ({len(research)} chars)")

        # Step 2: Initial Writing
        print(f"\n[2] WRITER is creating initial draft...")
        draft = self.writer.run(topic, context=research)
        print(f"    Done. ({len(draft)} chars)")

        # Step 3: Review Loop
        current_draft = draft
        revision = 0

        while revision < self.max_revisions:
            revision += 1
            print(f"\n[3.{revision}] CRITIC is reviewing (Revision {revision})...")

            review = self.critic.run(
                f"Review this content about: {topic}",
                context=current_draft
            )

            # Extract quality score
            quality = self._extract_quality_score(review)
            print(f"    Quality score: {quality}/10")

            if quality >= quality_threshold:
                print(f"    Quality meets threshold! No more revisions needed.")
                break

            # Revise based on feedback
            print(f"    Below threshold. WRITER is revising...")
            revision_context = f"Original draft:\n{current_draft}\n\nCritic's feedback:\n{review}"
            current_draft = self.writer.run(
                f"Revise the content about {topic} based on the critic's feedback. Address all issues raised.",
                context=revision_context
            )
            print(f"    Revision complete. ({len(current_draft)} chars)")

        print(f"\n{'='*60}")
        print(f"  COMPLETE after {revision} revision(s)")
        print(f"{'='*60}")

        return {
            "research": research,
            "initial_draft": draft,
            "final_draft": current_draft,
            "last_review": review,
            "revisions": revision,
            "final_quality": quality
        }

print("IterativeAgentSystem defined!")

In [ ]:
# 4.2 Run the iterative system

iterative_system = IterativeAgentSystem(
    researcher=researcher,
    writer=summarizer,
    critic=critic,
    llm=llm,
    max_revisions=2
)

results = iterative_system.run(
    topic="The impact of Agentic AI on software development workflows",
    quality_threshold=7
)

In [ ]:
# 4.3 View the improvement from initial to final draft
print("=== INITIAL DRAFT (first 500 chars) ===")
print(results["initial_draft"][:500])
print("\n" + "="*50)
print("\n=== FINAL DRAFT (first 500 chars) ===")
print(results["final_draft"][:500])
print(f"\nRevisions made: {results['revisions']}")
print(f"Final quality score: {results['final_quality']}/10")

---
## Part 5: Hierarchical Multi-Agent (Boss-Worker Pattern) (Hands-On - 10 min)

Build a system where an orchestrator agent decides which worker to delegate to.

In [ ]:
# 5.1 Hierarchical Orchestrator

class HierarchicalOrchestrator:
    """Boss agent that delegates tasks to worker agents."""

    def __init__(self, workers: dict, llm):
        self.workers = workers  # {name: Agent}
        self.llm = llm

    def run(self, request: str) -> str:
        """Process a request by delegating to appropriate workers."""
        # Step 1: Plan the work
        worker_list = "\n".join([f"- {name}: {agent.role}" for name, agent in self.workers.items()])

        plan_prompt = f"""You are a task orchestrator. Given a user request, create a plan.

Available workers:
{worker_list}

User request: {request}

Create a step-by-step plan. For each step, specify:
1. Which worker to use (exact name from the list)
2. What task to give them

Return JSON array format:
[{{"worker": "worker_name", "task": "specific task description"}}]

Return ONLY the JSON array, nothing else."""

        plan_response = self.llm.invoke(plan_prompt)

        # Parse plan (with fallback)
        try:
            # Extract JSON from response
            response_text = plan_response.content if hasattr(plan_response, 'content') else str(plan_response)
            # Find JSON array in response
            import re
            json_match = re.search(r'\[.*\]', response_text, re.DOTALL)
            if json_match:
                plan = json.loads(json_match.group())
            else:
                plan = [{"worker": list(self.workers.keys())[0], "task": request}]
        except (json.JSONDecodeError, IndexError):
            plan = [{"worker": list(self.workers.keys())[0], "task": request}]

        print(f"\nOrchestrator Plan ({len(plan)} steps):")
        for i, step in enumerate(plan, 1):
            print(f"  Step {i}: {step.get('worker', 'unknown')} -- {step.get('task', 'unknown')[:60]}")

        # Step 2: Execute plan
        accumulated_output = ""

        for i, step in enumerate(plan):
            worker_name = step.get("worker", "")
            task = step.get("task", "")

            if worker_name in self.workers:
                print(f"\n  Executing Step {i+1}: {worker_name}...")
                result = self.workers[worker_name].run(task, context=accumulated_output)
                accumulated_output += f"\n\n[{worker_name} output]:\n{result}"
                print(f"  Done. ({len(result)} chars)")
            else:
                print(f"  Warning: Worker '{worker_name}' not found, skipping.")

        return accumulated_output

# Create orchestrator with workers
orchestrator = HierarchicalOrchestrator(
    workers={
        "Researcher": researcher,
        "Summarizer": summarizer,
        "Critic": critic,
        "OutputGenerator": output_generator
    },
    llm=llm
)

print("Hierarchical Orchestrator created!")

In [ ]:
# 5.2 Test the hierarchical system
result = orchestrator.run(
    "Create a comprehensive guide on building AI applications with MCP (Model Context Protocol)"
)

# Show final output (last agent's output)
print("\n" + "="*60)
print("FINAL OUTPUT:")
print("="*60)
# Get the last section
sections = result.split("[")
if sections:
    print(sections[-1][:1000] if len(sections[-1]) > 1000 else sections[-1])

---
## Exercises

### Exercise 1: Custom Pipeline
Create a multi-agent pipeline for "Interview Preparation":
- Agent 1: Generate interview questions for a given role
- Agent 2: Provide model answers
- Agent 3: Create evaluation criteria
- Agent 4: Compile into a complete interview prep guide

### Exercise 2: Debate System
Create two agents: a Proponent (argues for) and an Opponent (argues against) a given topic. Run 3 rounds of debate and have a Judge agent determine the winner.

### Exercise 3: Add MCP Tools
Add MCP tools (calculator, web search) to the research agent. How does tool access improve the quality of research?

---

## Summary

### Key Takeaways:
1. **Multi-agent systems** solve complex problems through specialization and collaboration
2. **Sequential pipelines** are simple and effective for structured tasks
3. **Iterative refinement** with a critic agent significantly improves output quality
4. **Hierarchical orchestration** allows dynamic task delegation
5. Each agent should have a **clear role and well-defined system prompt**
6. Agent outputs flow as **context to the next agent** in the pipeline

### Next Session: Capstone Project + Deployment
Combine everything (RAG, agents, MCP) into a complete project with Streamlit UI.